In [35]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pedestrian-analysis-11-dec-2025/Pedestrian_Team-H_PC-02 (Balughat Bazar Road-23.82940369 90.39129782).xlsx
/kaggle/input/pedestrian-analysis-11-dec-2025/Pedestrain Count Survey Templet.xlsx
/kaggle/input/pedestrian-analysis-3-files-11-dec-2025/Pedestrian_Team-F_PC-04 (Kalshi Mor-23.82297769 90.37771611).xlsx
/kaggle/input/pedestrian-analysis-3-files-11-dec-2025/Pedestrian_Team-B_PC-03 (Avenue-11-Sagufta-23.837234 90.376174).xlsx
/kaggle/input/pedestrian-analysis-3-files-11-dec-2025/Pedestrian_Team-D_PC-05 (Jasimuddin Avenue-23.86141079 90.39406258).xlsx


In [36]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [43]:
import openpyxl
import datetime as dt  # add this import near the top of your file


# --------- file paths (edit these to your actual locations) ----------
template_path = r"/kaggle/input/pedestrian-analysis-11-dec-2025/Pedestrain Count Survey Templet.xlsx"
#data_path_2 = r"/kaggle/input/pedestrian-analysis-11-dec-2025/Pedestrian_Team-H_PC-02 (Balughat Bazar Road-23.82940369 90.39129782).xlsx"
data_path_3 = r"/kaggle/input/pedestrian-analysis-3-files-11-dec-2025/Pedestrian_Team-B_PC-03 (Avenue-11-Sagufta-23.837234 90.376174).xlsx"
#data_path_4 = r"/kaggle/input/pedestrian-analysis-3-files-11-dec-2025/Pedestrian_Team-F_PC-04 (Kalshi Mor-23.82297769 90.37771611).xlsx"
#data_path_5 = r"/kaggle/input/pedestrian-analysis-3-files-11-dec-2025/Pedestrian_Team-D_PC-05 (Jasimuddin Avenue-23.86141079 90.39406258).xlsx"
# --------- load the workbooks (keep formulas in template) -----------
tmpl_wb = openpyxl.load_workbook(template_path, data_only=False)
data_wb = openpyxl.load_workbook(data_path_3, data_only=True)

# --------- select the sheets we will use ----------------------------
tmpl_ws = tmpl_wb["Pedestrian"]        # template sheet
data_ws = data_wb["P1, P2, PC-1"]      # data sheet

print("Template sheet max row:", tmpl_ws.max_row)
print("Data sheet max row:", data_ws.max_row)

Template sheet max row: 95
Data sheet max row: 132


In [44]:
def get_time_rows(ws, time_col):
    """
    Return a list of row numbers where the given column
    contains an Excel time value (for example 7:00, 7:15, etc.).
    This automatically skips text rows like 'Sub-Total of one hour'.
    """
    rows = []
    for r in range(1, ws.max_row + 1):
        val = ws.cell(row=r, column=time_col).value
        if isinstance(val, dt.time):
            rows.append(r)
    return rows

# In both sheets, the 'Start' time is in column D = 4
tmpl_time_rows = get_time_rows(tmpl_ws, time_col=4)   # template
data_time_rows = get_time_rows(data_ws, time_col=4)   # data file

print("Template time rows:", len(tmpl_time_rows), tmpl_time_rows[:10])
print("Data time rows:", len(data_time_rows), data_time_rows[:10])

Template time rows: 64 [14, 15, 16, 17, 19, 20, 21, 22, 24, 25]
Data time rows: 64 [11, 12, 13, 14, 16, 17, 18, 19, 21, 22]


In [45]:
# --- Step 3: copy Direction-P1 & Direction-P2 into P1 & P2 ---

# Use the minimum length, just in case the two lists differ slightly
n = min(len(tmpl_time_rows), len(data_time_rows))
print("Number of rows to copy:", n) 

for i in range(n):
    tr = tmpl_time_rows[i]   # row in template
    dr = data_time_rows[i]   # corresponding row in data file

    # From data file:
    #   column 6 (F) = Direction-P1
    #   column 7 (G) = Direction-P2
    dir_p1 = data_ws.cell(row=dr, column=6).value
    dir_p2 = data_ws.cell(row=dr, column=7).value

    # To template:
    #   column 6 (F) = P1
    #   column 7 (G) = P2
    tmpl_ws.cell(row=tr, column=6).value = dir_p1
    tmpl_ws.cell(row=tr, column=7).value = dir_p2

print("Copy completed for", n, "time rows.")


Number of rows to copy: 64
Copy completed for 64 time rows.


In [46]:
# --- Step 5: copy Direction-PC-1 into PC-1 (Q) and set PC-2 (R) = 0 ---

for i in range(n):                 # same n, tmpl_time_rows, data_time_rows as before
    tr = tmpl_time_rows[i]         # row in template
    dr = data_time_rows[i]         # corresponding row in data sheet

    # Data sheet: column 8 (H) = Direction-PC-1
    pc1_val = data_ws.cell(row=dr, column=8).value

    # Template sheet:
    #   column 17 (Q) = PC-1
    #   column 18 (R) = PC-2
    tmpl_ws.cell(row=tr, column=17).value = pc1_val   # PC-1
    tmpl_ws.cell(row=tr, column=18).value = 0         # PC-2 = 0

print("PC-1 and PC-2 columns filled for", n, "time rows.")


PC-1 and PC-2 columns filled for 64 time rows.


In [47]:
# --- Step 6 (revised): Update Day and Date for ALL rows in the data block ---

new_day = "Thursday"
new_date = "11.20.2025"

# Find the vertical range where our table lives
start_row = min(tmpl_time_rows)
end_row = max(tmpl_time_rows)

for r in range(start_row, end_row + 1):
    # Columns:
    # B = 2 (Day)
    # C = 3 (Date)
    # M = 13 (Day)
    # N = 14 (Date)

    tmpl_ws.cell(row=r, column=2).value = new_day     # B: Day
    tmpl_ws.cell(row=r, column=3).value = new_date    # C: Date
    tmpl_ws.cell(row=r, column=13).value = new_day    # M: Day
    tmpl_ws.cell(row=r, column=14).value = new_date   # N: Date

print("Day and Date updated for ALL rows from", start_row, "to", end_row)


Day and Date updated for ALL rows from 14 to 92


In [49]:
# --- Step 4: save the updated template as a new file ---

#output_path_2 = r"Pedestrian_Count_Filled_PC02.xlsx"   # you can change the name/path
output_path_3 = r"Pedestrian_Count_Filled_PC03.xlsx"
#output_path_4 = r"Pedestrian_Count_Filled_PC04.xlsx"
#output_path_5 = r"Pedestrian_Count_Filled_PC05.xlsx"

tmpl_wb.save(output_path)
print("Saved updated workbook as:", output_path_3)


Saved updated workbook as: Pedestrian_Count_Filled_PC03.xlsx
